In [42]:
# =====================================================================
# SYSTEM INITIALIZATION & DEPENDENCY SETUP (UPDATED FOR DATABASE TRACKING)
# =====================================================================
!pip install mlflow pydantic xgboost -q

import mlflow
import pydantic
import sklearn
import pandas as pd
import numpy as np

# Enterprise Solution: Route metadata tracking through a structured local SQLite DB
# This completely avoids the file-store maintenance exception
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("FinTech_Credit_Risk_Architecture")

print("--- System Audit Complete ---")
print(f"MLflow Registry: Active (SQLite Database Engine)")
print(f"Pydantic Version: {pydantic.__version__}")
print(f"Scikit-Learn Version: {sklearn.__version__}")

2026/06/14 10:19:49 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/14 10:19:49 INFO mlflow.store.db.utils: Updating database tables
2026/06/14 10:19:51 INFO mlflow.tracking.fluent: Experiment with name 'FinTech_Credit_Risk_Architecture' does not exist. Creating a new experiment.


--- System Audit Complete ---
MLflow Registry: Active (SQLite Database Engine)
Pydantic Version: 2.12.3
Scikit-Learn Version: 1.6.1


# Enterprise Credit Risk & Default Prediction System

### 1. Corporate Strategy & Business Context
In consumer lending, misclassifying a high-risk borrower who will eventually default (a False Negative) costs banks significantly more than passing up a conservative borrower (a False Positive).

### 2. Technical Blueprint
This notebook establishes a transparent, audited risk-scoring model using the classic financial credit simulation dataset. To prepare for our future multi-container FastAPI platform, we implement:
* **Strict Runtime Boundaries:** A Pydantic schema enforcing validation guards on raw input streams.
* **Lineage Tracking:** An instrumented MLflow wrapper logging model hyperparameters, data shapes, and evaluation metrics automatically into a SQL storage registry.

In [44]:
# =====================================================================
# DATA ACQUISITION & PRODUCTION TRAIN/TEST SEPARATION
# =====================================================================
import pandas as pd
from sklearn.model_selection import train_test_split

# Ingest an evergreen credit simulation dataset containing missing fields and complex categorical data
dataset_url = "https://raw.githubusercontent.com/PhilChodrow/ml-notes/main/data/credit-risk/credit_risk_dataset.csv"
raw_credit_data = pd.read_csv(dataset_url)

print("--- Raw Financial Data Structural Audit ---")
print(f"Total Applications Profiled: {raw_credit_data.shape[0]}")
print(f"Total Financial Columns: {raw_credit_data.shape[1]}")

# Isolate target variable 'loan_status' (1 = Default, 0 = Non-Default)
X = raw_credit_data.drop(columns=['loan_status'])
y = raw_credit_data['loan_status']

# Split data using stratification to preserve the credit default distribution ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training Partition Base Shape: {X_train.shape}")
print(f"Testing Partition Base Shape: {X_test.shape}")

--- Raw Financial Data Structural Audit ---
Total Applications Profiled: 32581
Total Financial Columns: 12
Training Partition Base Shape: (26064, 11)
Testing Partition Base Shape: (6517, 11)


## 3. Designing the Live API Validation Contract (Pydantic)
Before writing any data science code, we establish our strict **Data Gateway**. This Pydantic model defines the exact structural format the Flask UI must send, and the exact type rules the FastAPI backend will enforce at runtime. If a user tries to submit an age under 18 or an impossible loan amount, the API drops the payload before it can corrupt the model.

In [45]:
# =====================================================================
# PYDANTIC RUNTIME DATA VALIDATION GATEWAY
# =====================================================================
from pydantic import BaseModel, Field, field_validator
from typing import Literal

class CreditApplicationSchema(BaseModel):
    # Enforce type constraints and realistic financial boundaries
    person_age: int = Field(..., ge=18, le=120, description="Borrower must be a legal adult.")
    person_income: float = Field(..., ge=0.0)
    person_home_ownership: Literal['RENT', 'MORTGAGE', 'OWN', 'OTHER']
    person_emp_length: float = Field(..., ge=0.0, le=60.0)
    loan_intent: Literal['PERSONAL', 'EDUCATION', 'MEDICAL', 'VENTURE', 'HOMEIMPROVEMENT', 'DEBTCONSOLIDATION']
    loan_grade: Literal['A', 'B', 'C', 'D', 'E', 'F', 'G']
    loan_amnt: float = Field(..., ge=500.0, le=100000.0)
    loan_int_rate: float = Field(..., ge=0.0, le=50.0)
    loan_percent_income: float = Field(..., ge=0.0, le=1.0)
    cb_person_default_on_file: Literal['Y', 'N']
    cb_person_cred_hist_length: int = Field(..., ge=0.0)

    @field_validator('person_income', 'loan_amnt')
    @classmethod
    def enforce_nonzero_economics(cls, value: float) -> float:
        if value <= 0:
            raise ValueError("Financial parameters must represent positive numeric values.")
        return value

print("Pydantic Credit Application Validation Contract safely compiled!")

Pydantic Credit Application Validation Contract safely compiled!


## 4. Multi-Branch Column Transformations
The credit dataset contains realistic business anomalies: `person_emp_length` and `loan_int_rate` contain missing elements (NaNs). We decouple our transformations using an immutable `ColumnTransformer` pipeline. This approach cleanly scales numerical features and uses categorical flags to handle categorical text without messing with the data structure.

In [46]:
# =====================================================================
# BULLETPROOF COLUMNTRANSFORMER SEPARATION PIPELINE
# =====================================================================
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Classify operational processing routes based on our data schema
numerical_features = [
    'person_age', 'person_income', 'person_emp_length',
    'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length'
]
categorical_features = [
    'person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file'
]

# Impute and standardize our numerical features safely
numerical_sub_pipeline = Pipeline(steps=[
    ('median_imputer', SimpleImputer(strategy='median')),
    ('feature_scaler', StandardScaler())
])

# Gracefully handle missing strings and encode categories safely for live requests
categorical_sub_pipeline = Pipeline(steps=[
    ('missing_imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('one_hot_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first'))
])

# Fuse the parallel matrix channels together
production_preprocessing_gate = ColumnTransformer(transformers=[
    ('num_track', numerical_sub_pipeline, numerical_features),
    ('cat_track', categorical_sub_pipeline, categorical_features)
])

print("Preprocessing framework securely bound. Ready for model insertion.")

Preprocessing framework securely bound. Ready for model insertion.


## 5. Instrumented Model Training & Audit Trailing (MLflow)
Now, we bind the preprocessing gate to an advanced gradient-boosted classification engine (**XGBoost**). Instead of training the model undocumented, we launch an active **MLflow tracking block**. This block acts like a flight data recorder—automatically capturing training patterns, tuning parameters, and credit default matrices directly inside our local SQLite instance.

In [47]:
# =====================================================================
# INSTRUMENTED TRAINING LOOP & RECOVERY ENGINE
# =====================================================================
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
import mlflow.sklearn

# Calculate the precise class imbalance weight (~4.0x ratio of non-default to default)
imbalance_multiplier = float(y_train.value_counts()[0] / y_train.value_counts()[1])

# Build the complete operational blueprint
credit_risk_pipeline = Pipeline(steps=[
    ('preprocessing_gate', production_preprocessing_gate),
    ('xgboost_engine', XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.04,
        scale_pos_weight=imbalance_multiplier,
        random_state=42,
        n_jobs=-1
    ))
])

# Trigger MLflow Autologging to catch all Scikit-Learn and XGBoost telemetry natively
mlflow.sklearn.autolog(log_models=True)

print("Starting training session. Auditing live metrics with MLflow...")
with mlflow.start_run(run_name="Credit_Risk_XGBoost_Production") as active_run:

    # Train the end-to-end model pipeline
    credit_risk_pipeline.fit(X_train, y_train)

    # Generate validation sets for auditing
    predictions = credit_risk_pipeline.predict(X_test)
    probabilities = credit_risk_pipeline.predict_proba(X_test)[:, 1]

    # Compute enterprise performance evaluations
    final_roc_auc = roc_auc_score(y_test, probabilities)

    # Explicitly log high-value risk metrics to our MLflow dashboard
    mlflow.log_metric("test_roc_auc_score", final_roc_auc)

    print("\n=====================================================================")
    print("--- MLFLOW TRANSACTION TRACE COMPLETE ---")
    print(f"Logged SQL Run ID: {active_run.info.run_id}")
    print(f"Production ROC-AUC Performance Metric: {final_roc_auc:.4f}")
    print("=====================================================================\n")
    print(classification_report(y_test, predictions))

Starting training session. Auditing live metrics with MLflow...


2026/06/14 10:26:22 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/06/14 10:26:23 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.12/dist-packages/mlflow/types/utils


--- MLFLOW TRANSACTION TRACE COMPLETE ---
Logged SQL Run ID: 8c3053f39d6d48b180836f29a3789d45
Production ROC-AUC Performance Metric: 0.9371

              precision    recall  f1-score   support

           0       0.94      0.94      0.94      5095
           1       0.78      0.78      0.78      1422

    accuracy                           0.90      6517
   macro avg       0.86      0.86      0.86      6517
weighted avg       0.90      0.90      0.90      6517



In [48]:
# =====================================================================
# PRODUCTION EXPORT PROTOCOL
# =====================================================================
from google.colab import files
import joblib

# Save the complete pipeline asset (contains imputer parameters, encoders, and weights)
joblib.dump(credit_risk_pipeline, 'credit_risk_pipeline.pkl')

# Trigger direct local machine file-download
files.download('credit_risk_pipeline.pkl')
print("Successfully generated and downloaded 'credit_risk_pipeline.pkl'!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Successfully generated and downloaded 'credit_risk_pipeline.pkl'!
